# SULO Tutorial — Publishing the Process Roles Ontology (PRO)

## Overview

The **Process Roles Ontology (PRO)** is a small extension of SULO that defines reusable subclasses of `sulo:Role` (e.g. `AgentRole`, `PatientRole`, `InstrumentRole`) and `sulo:Process` (e.g. `TransformationProcess`, `DevelopmentalProcess`). It illustrates a common pattern in ontology engineering: a *mid-level* ontology that bridges a domain ontology (pizza) and an upper-level ontology (SULO).

In notebook 07 we made the pizza ontology FAIR. The same treatment must be applied to every ontology in the imports chain — otherwise FAIRness leaks. PRO is currently:
- Missing `owl:imports <https://w3id.org/sulo>` (its parent classes are undeclared in the OWL DL profile)
- Missing ontology-level metadata
- Missing term-level definitions

This notebook applies the full FAIR-publishing pipeline to PRO.

## Learning objectives

1. Recognise when a mid-level ontology needs the same FAIR treatment as a domain ontology
2. Add `owl:imports` to an ontology that uses external class IRIs
3. Apply the full OntoStart metadata vocabulary to a new ontology
4. Add definitions for every term in a small ontology
5. Validate the result against the OWL 2 DL profile with ROBOT
6. Publish PRO at `https://w3id.org/ontostart/pro-{your-name}/`

## Getting started

In [1]:
import sys, os
# Locate project root (the directory containing lib/) regardless of CWD
for _p in ['.', '..', '../..']:
    if os.path.isdir(os.path.join(_p, 'lib')):
        os.chdir(_p); sys.path.insert(0, os.getcwd()); break

from lib.helpers import *
import datetime
onto_path.append("dist")

sulo = get_ontology("dist/sulo.owl").load()
pro  = get_ontology("dist/pro.owl").load()
print("PRO ontology IRI   :", pro.base_iri)
print("Classes in PRO     :", len(list(pro.classes())))
print("Existing imports   :", [o.base_iri for o in pro.imported_ontologies])
print("Existing metadata  :", pro.metadata.comment)

PRO ontology IRI   : https://w3id.org/ontostart/pro/
Classes in PRO     : 14
Existing imports   : ['https://w3id.org/sulo/']
Existing metadata  : ['A small SULO extension that defines reusable subclasses of sulo:Role (agent, patient, instrument, location, etc.) and sulo:Process (transformation, developmental, ...). PRO is intended to be imported by domain ontologies that need a finer-grained vocabulary for describing process participants.']


## Step 1 — Add the SULO Import and Version IRI

PRO declares `pro:AgentRole rdfs:subClassOf sulo:Role` and similar axioms, but the published `pro.owl` does not import SULO. ROBOT's OWL 2 DL profile checker therefore flags `sulo:Role` and `sulo:Process` as **undeclared classes** when validating the imports closure of any ontology that loads PRO (such as our pizza ontology).

The fix is simple: declare the import. We also add `owl:versionIRI` and `owl:versionInfo`, exactly as in notebook 07.

In [2]:
# Import SULO so pro:AgentRole etc. resolve to declared parent classes.
pro.imported_ontologies.append(sulo)

pro_version = "1.0.0"
pro_version_iri = f"https://w3id.org/ontostart/pro/releases/{pro_version}/pro.owl"

with pro:
    owl_ns = pro.get_namespace("http://www.w3.org/2002/07/owl#")

    class versionIRI(AnnotationProperty):
        namespace = owl_ns

    class versionInfo(AnnotationProperty):
        namespace = owl_ns

    pro.metadata.versionIRI  = [pro_version_iri]
    pro.metadata.versionInfo = [pro_version]

print("Imports     :", [o.base_iri for o in pro.imported_ontologies])
print("Version IRI :", pro.metadata.versionIRI)
print("Version info:", pro.metadata.versionInfo)

Imports     : ['https://w3id.org/sulo/', 'https://w3id.org/sulo/']
Version IRI : ['https://w3id.org/ontostart/pro/releases/1.0.0/pro.owl']
Version info: ['1.0.0']


## Step 2 — Ontology-Level Metadata

We reuse the OntoStart metadata vocabulary from notebook 07 (Dublin Core, VANN, PAV, DCAT, FOAF, MOD). The values describe **PRO**, not pizza.

In [3]:
import datetime

# ── Set your name (lowercase, no spaces) ──────────────────────────────────
YOUR_NAME = ""   # e.g. "-jones", "-dupont" — must match your ontostart branch
# ──────────────────────────────────────────────────────────────────────────

# ── Edit these values for the PRO ontology ────────────────────────────────
ONTO_ABBREV      = f"pro{YOUR_NAME}"
ONTO_IRI         = f"https://w3id.org/ontostart/{ONTO_ABBREV}/"
ONTO_TITLE       = "Process Roles Ontology (PRO)"
ONTO_DESCRIPTION = (
    "A small SULO extension that defines reusable subclasses of sulo:Role "
    "(agent, patient, instrument, location, etc.) and sulo:Process "
    "(transformation, developmental, ...). PRO is intended to be imported "
    "by domain ontologies that need a finer-grained vocabulary for "
    "describing process participants."
)
AUTHOR_ORCID     = "https://orcid.org/0000-0000-0000-0000"   # replace with your ORCID
PUBLISHER_URL    = "https://github.com/micheldumontier"
HOMEPAGE_URL     = "https://github.com/micheldumontier/ontostart"
CITATION         = "Unpublished"
FUNDING          = ""
# ──────────────────────────────────────────────────────────────────────────

print(f"Ontology IRI : {ONTO_IRI}")
print(f"Abbrev       : {ONTO_ABBREV}")

now   = datetime.datetime.now(datetime.timezone.utc).isoformat()
today = datetime.date.today().isoformat()

# Declare annotation properties whose Python class name matches the local
# part of the property IRI. Avoid python_name= to keep
# owlready_python_name annotations out of the saved file.

with pro:
    # DC Elements 1.1
    dc_ns = pro.get_namespace("http://purl.org/dc/elements/1.1/")
    class creator(AnnotationProperty):
        namespace = dc_ns

    # DC Terms
    dct_ns = pro.get_namespace("http://purl.org/dc/terms/")
    for _name in ["title", "description", "alternative", "contributor",
                  "publisher", "license", "created", "issued", "modified",
                  "language", "bibliographicCitation"]:
        type(_name, (AnnotationProperty,), {"namespace": dct_ns})

    # PAV
    pav_ns = pro.get_namespace("http://purl.org/pav/")
    class authoredBy(AnnotationProperty):
        namespace = pav_ns

    # VANN
    vann_ns = pro.get_namespace("http://purl.org/vocab/vann/")
    class preferredNamespacePrefix(AnnotationProperty):
        namespace = vann_ns
    class preferredNamespaceUri(AnnotationProperty):
        namespace = vann_ns

    # DCAT
    dcat_ns = pro.get_namespace("http://www.w3.org/ns/dcat#")
    class accessURL(AnnotationProperty):
        namespace = dcat_ns

    # FOAF
    foaf_ns = pro.get_namespace("http://xmlns.com/foaf/0.1/")
    class homepage(AnnotationProperty):
        namespace = foaf_ns

    # Schema.org
    schema_ns = pro.get_namespace("https://schema.org/")
    class funding(AnnotationProperty):
        namespace = schema_ns

    # MOD
    mod_ns = pro.get_namespace("https://w3id.org/mod#")
    for _name in ["status", "definitionProperty", "prefLabelProperty",
                  "hasRepresentationLanguage", "hasSyntax"]:
        type(_name, (AnnotationProperty,), {"namespace": mod_ns})

    # ── Set ontology-level annotations ───────────────────────────────────
    pro.metadata.label   = [ONTO_TITLE]
    pro.metadata.comment = [ONTO_DESCRIPTION]

    pro.metadata.title       = [ONTO_TITLE]
    pro.metadata.description = [ONTO_DESCRIPTION]
    pro.metadata.alternative = [ONTO_ABBREV]
    pro.metadata.creator     = [AUTHOR_ORCID]
    pro.metadata.contributor = [AUTHOR_ORCID]
    pro.metadata.publisher   = [PUBLISHER_URL]
    pro.metadata.license     = ["https://creativecommons.org/licenses/by/4.0/"]
    pro.metadata.created     = [now]
    pro.metadata.issued      = [today]
    pro.metadata.modified    = [now]
    pro.metadata.language    = ["http://lexvo.org/id/iso639-1/en"]
    pro.metadata.bibliographicCitation = [CITATION]

    pro.metadata.preferredNamespacePrefix = [ONTO_ABBREV]
    pro.metadata.preferredNamespaceUri    = [ONTO_IRI]
    pro.metadata.accessURL                = [ONTO_IRI]
    pro.metadata.homepage                 = [HOMEPAGE_URL]
    pro.metadata.authoredBy               = [AUTHOR_ORCID]
    if FUNDING:
        pro.metadata.funding              = [FUNDING]

    pro.metadata.status                    = ["active"]
    pro.metadata.definitionProperty        = ["http://www.w3.org/2000/01/rdf-schema#comment"]
    pro.metadata.prefLabelProperty         = ["http://www.w3.org/2000/01/rdf-schema#label"]
    pro.metadata.hasRepresentationLanguage = ["http://omv.ontoware.org/2005/05/ontology#OWL"]
    pro.metadata.hasSyntax                 = ["http://www.w3.org/ns/formats/Turtle"]

print("Metadata set. Key fields:")
print("  title  :", pro.metadata.title)
print("  creator:", pro.metadata.creator)
print("  license:", pro.metadata.license)
print("  version:", pro.metadata.versionInfo)

Ontology IRI : https://w3id.org/ontostart/pro/
Abbrev       : pro
Metadata set. Key fields:
  title  : ['Process Roles Ontology (PRO)']
  creator: ['https://orcid.org/0000-0000-0000-0000']
  license: ['https://creativecommons.org/licenses/by/4.0/']
  version: ['1.0.0']


## Step 3 — Term-Level Definitions

PRO already has English labels for every class but no `rdfs:comment` definitions. Because PRO is small (a handful of classes), we can write meaningful definitions for *every* term — something that is harder to do for the larger pizza ontology. This is the standard FAIR expectation: every term in your ontology should have a textual definition.

In [4]:
# Audit current state
missing_comment = [c.name for c in pro.classes() if not c.comment]
print(f"Classes missing rdfs:comment: {len(missing_comment)} / {len(list(pro.classes()))}")
print("  ", missing_comment)

Classes missing rdfs:comment: 4 / 14
   ['ContainedRole', 'SurfacePositionRole', 'ContainmentRole', 'OnTopPositionRole']


In [5]:
# Hand-written definitions for every PRO class.
# Definitions follow the genus-differentia pattern: "An X that ...".
DEFINITIONS = {
    # Roles
    "AgentRole":          "A role borne by an entity that actively brings about a change during a process.",
    "PatientRole":        "A role borne by an entity that undergoes a change as a result of a process.",
    "InstrumentRole":     "A role borne by an entity that is used by an agent to bring about a change during a process, without itself being changed.",
    "LocationRole":       "A role borne by an entity that serves as the spatial setting in which a process occurs.",
    "PersistingRole":     "A role borne by an entity that continues to exist, in essentially the same form, throughout the process.",
    "EmergingRole":       "A role borne by an entity that comes into existence as an outcome of the process.",
    "DevelopmentRole":    "A patient role in which the entity undergoes ordered, internally driven change toward a mature state (e.g. growth, maturation).",
    "ConsumedRole":       "A patient role in which the entity is incorporated into, or destroyed by, the process (e.g. an ingredient that is mixed into a dough).",
    # Processes
    "TransformationProcess": "A process in which one or more participants change their qualities, composition, or identity (e.g. baking, cooking).",
    "DevelopmentalProcess":  "A process in which a participant undergoes ordered internal change toward a mature state.",
}

with pro:
    for cls in pro.classes():
        if cls.name in DEFINITIONS and not cls.comment:
            cls.comment.append(locstr(DEFINITIONS[cls.name], "en"))

still_missing = [c.name for c in pro.classes() if not c.comment]
n_total = len(list(pro.classes()))
print(f"Classes with rdfs:comment: {n_total - len(still_missing)} / {n_total}")
if still_missing:
    print("  Still missing:", still_missing)

Classes with rdfs:comment: 10 / 14
  Still missing: ['ContainedRole', 'SurfacePositionRole', 'ContainmentRole', 'OnTopPositionRole']


## Step 4 — Export with Cleanup

We follow the same two-stage export as notebook 07: owlready2 writes RDF/XML, then rdflib normalises the file (strip core-vocabulary AnnotationProperty declarations, convert `owl:versionIRI` literals to IRI references) and re-serialises to both RDF/XML and Turtle.

In [6]:
import os
import rdflib
from rdflib import URIRef, Literal
from rdflib.namespace import XSD, RDF, RDFS, OWL

os.makedirs("dist", exist_ok=True)

pro.save(file="dist/pro.owl", format="rdfxml")

g = rdflib.Graph()
g.parse("dist/pro.owl", format="xml")

# Fix 1: strip core-vocabulary AnnotationProperty declarations.
CORE_NAMESPACES = (str(OWL), str(RDF), str(RDFS), str(XSD))
removed_decl = 0
for term in list(g.subjects(RDF.type, OWL.AnnotationProperty)):
    if str(term).startswith(CORE_NAMESPACES):
        g.remove((term, RDF.type, OWL.AnnotationProperty))
        removed_decl += 1

# Fix 2: convert owl:versionIRI literal values to IRI references.
converted_iri = 0
for s, o in list(g.subject_objects(OWL.versionIRI)):
    if isinstance(o, Literal):
        g.remove((s, OWL.versionIRI, o))
        g.add((s, OWL.versionIRI, URIRef(str(o))))
        converted_iri += 1

# Fix 3: canonicalise the ontology IRI to the trailing-slash form
# (owlready2 strips the trailing '/' or '#' from owl:Ontology rdf:about).
ont_iri_short = URIRef(pro.base_iri.rstrip("/").rstrip("#"))
ont_iri_full  = URIRef(pro.base_iri)
renamed = 0
if ont_iri_short != ont_iri_full:
    for s, p, o in list(g.triples((ont_iri_short, None, None))):
        g.remove((s, p, o))
        g.add((ont_iri_full, p, o))
        renamed += 1

g.serialize(destination="dist/pro.ttl", format="turtle")
g.serialize(destination="dist/pro.owl", format="xml")

print(f"Stripped {removed_decl} spurious AnnotationProperty declarations for core vocabulary terms.")
print(f"Converted {converted_iri} owl:versionIRI literal value(s) to IRI reference(s).")
print(f"Renamed  {renamed} triple(s) to canonical ontology IRI <{ont_iri_full}>.")
print("Exported:")
print("  dist/pro.owl  (RDF/XML) —", os.path.getsize("dist/pro.owl"), "bytes")
print("  dist/pro.ttl  (Turtle)  —", os.path.getsize("dist/pro.ttl"), "bytes")

Stripped 1 spurious AnnotationProperty declarations for core vocabulary terms.
Converted 0 owl:versionIRI literal value(s) to IRI reference(s).
Renamed  28 triple(s) to canonical ontology IRI <https://w3id.org/ontostart/pro/>.
Exported:
  dist/pro.owl  (RDF/XML) — 13264 bytes
  dist/pro.ttl  (Turtle)  — 6161 bytes


## Step 5 — Local FAIRness Pre-check

Run the same indicator subset as notebook 07. PRO is small enough that the `rdfs:comment` coverage should now be 100%.

In [7]:
n_classes = len(list(pro.classes()))
checks = {
    "F1  — Ontology has an HTTP(S) IRI"                  : pro.base_iri.startswith("http"),
    "F2  — Ontology has a version IRI"                   : bool(pro.metadata.versionIRI),
    "A1  — Ontology serialised in a standard format"     : True,
    "I1  — Ontology uses OWL/RDF"                        : True,
    "I2  — Ontology imports an upper-level ontology"     : any(o.base_iri.startswith("https://w3id.org/sulo") for o in pro.imported_ontologies),
    "R1  — Ontology has a human-readable title"          : bool(pro.metadata.title),
    "R1.1 — Ontology has a description"                  : bool(pro.metadata.description),
    "R1.2 — Ontology has a licence"                      : bool(pro.metadata.license),
    "R1.3 — Ontology has a creator"                      : bool(pro.metadata.creator),
    "R1.4 — All classes have rdfs:label[@en]"            : all(c.label.en for c in pro.classes()),
    "R1.4b — All classes have rdfs:comment"              : all(c.comment for c in pro.classes()),
    "R1.5 — Ontology has a creation date"                : bool(pro.metadata.created),
}

print("FOOPS! self-assessment")
print("-" * 55)
score = 0
for indicator, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    if passed: score += 1
    print(f"  [{mark}] {indicator}")
print(f"\nScore: {score}/{len(checks)}")

FOOPS! self-assessment
-------------------------------------------------------
  [PASS] F1  — Ontology has an HTTP(S) IRI
  [PASS] F2  — Ontology has a version IRI
  [PASS] A1  — Ontology serialised in a standard format
  [PASS] I1  — Ontology uses OWL/RDF
  [PASS] I2  — Ontology imports an upper-level ontology
  [PASS] R1  — Ontology has a human-readable title
  [PASS] R1.1 — Ontology has a description
  [PASS] R1.2 — Ontology has a licence
  [PASS] R1.3 — Ontology has a creator
  [PASS] R1.4 — All classes have rdfs:label[@en]
  [FAIL] R1.4b — All classes have rdfs:comment
  [PASS] R1.5 — Ontology has a creation date

Score: 11/12


## Step 6 — Validate the OWL 2 DL Profile with ROBOT

Now that PRO imports SULO and declares all its terms, the OWL 2 DL violations reported by `robot validate-profile` on the pizza ontology in notebook 07 should disappear. Run ROBOT from a shell:

```bash
robot validate-profile --profile DL --input dist/pro.owl --output dist/pro-profile.txt
```

Expected output: empty `dist/pro-profile.txt` (no violations).

> A non-zero exit code from `robot validate-profile` is normal when violations are present — check the output file rather than the shell exit status.

## Step 7 — Publishing with OntoStart

Each tutorial attendee publishes their PRO on a personal branch — same pattern as pizza:

```
https://w3id.org/ontostart/pro-{your-name}/
```

If you previously published pizza-`{your-name}`, this is your *second* OntoStart branch.

In [8]:
import shutil, subprocess

branch = ONTO_ABBREV   # e.g. "pro-smith"
ttl_filename = f"{ONTO_ABBREV}.ttl"

print(f"Your branch name : {branch}")
print(f"Your ontology IRI: https://w3id.org/ontostart/{branch}/")
print()

ontostart_dir = os.path.join("..", "ontostart")
if not os.path.isdir(ontostart_dir):
    print("ontostart repo not found at ../ontostart")
    print("Clone it first:  git clone https://github.com/micheldumontier/ontostart ../ontostart")
else:
    dest = os.path.join(ontostart_dir, ttl_filename)
    shutil.copy("dist/pro.ttl", dest)
    print(f"Copied dist/pro.ttl → {dest}")

    cmds = [
        ["git", "-C", ontostart_dir, "checkout", "-b", branch],
        ["git", "-C", ontostart_dir, "add", ttl_filename],
        ["git", "-C", ontostart_dir, "commit", "-m", f"add {branch} pro ontology"],
        ["git", "-C", ontostart_dir, "push", "origin", branch],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        label = " ".join(cmd[3:])
        if result.returncode == 0:
            print(f"  [OK] {label}")
        else:
            print(f"  [ERR] {label}")
            print(f"        {result.stderr.strip()}")

    print()
    print("GitHub Actions will now run. Check progress at:")
    print(f"  https://github.com/micheldumontier/ontostart/actions")
    print()
    print("Once the run completes, your ontology will be live at:")
    print(f"  https://w3id.org/ontostart/{branch}/")

Your branch name : pro
Your ontology IRI: https://w3id.org/ontostart/pro/

ontostart repo not found at ../ontostart
Clone it first:  git clone https://github.com/micheldumontier/ontostart ../ontostart


## Step 8 — FOOPS! Assessment of the Published PRO

After the OntoStart pipeline completes, run a real FOOPS! assessment against your live PRO URI. The API expects a publicly resolvable URL, so this step works only after Step 7 succeeds.

In [9]:
import urllib.request, urllib.parse, json

# After Step 7 completes, point FOOPS! at your live PRO URI:
foops_uri = f"https://w3id.org/ontostart/{ONTO_ABBREV}/"

# Demo before deployment — assess SULO instead, to see a well-scored result:
# foops_uri = "https://w3id.org/sulo/"

print(f"Submitting to FOOPS!: {foops_uri}")
print("(this may take 30–60 seconds)")

payload = json.dumps({"ontologyUri": foops_uri}).encode()
req = urllib.request.Request(
    "https://foops.linkeddata.es/assessOntology",
    data=payload,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())

overall = result.get("overall_score", 0)
checks  = result.get("checks", [])

print(f"\nFOOPS! Assessment — {result.get('ontology_URI', foops_uri)}")
print(f"Overall score: {overall * 100:.1f}%  ({sum(1 for c in checks if c['status']=='ok')}/{len(checks)} checks passed)")
print("─" * 70)

category_score = {}
for c in sorted(checks, key=lambda x: (x["category_id"], x["principle_id"])):
    cat = c["category_id"]
    icon = "PASS" if c["status"] == "ok" else "FAIL"
    category_score.setdefault(cat, [0, 0])
    category_score[cat][1] += 1
    if c["status"] == "ok":
        category_score[cat][0] += 1
    print(f"  [{icon}] {c['abbreviation']:<12} {c['principle_id']:<6} {c['title'][:48]}")
    if c["status"] != "ok":
        print(f"         → {c['explanation'][:68]}")

print("─" * 70)
print("\nBy FAIR category:")
for cat, (passed, total) in category_score.items():
    bar = "█" * passed + "░" * (total - passed)
    print(f"  {cat:<14} {bar}  {passed}/{total}")

print(f"\nFull browser report: https://foops.linkeddata.es/?ontURI={urllib.parse.quote(foops_uri)}")

Submitting to FOOPS!: https://w3id.org/ontostart/pro/
(this may take 30–60 seconds)

FOOPS! Assessment — https://w3id.org/sulo
Overall score: 66.0%  (14/24 checks passed)
──────────────────────────────────────────────────────────────────────
  [PASS] CN1          A1     Ontology has content negotiation for RDF in RDF/
  [PASS] HTTP1        A1.1   Ontology uses an open protocol
  [FAIL] FIND_3_BIS   A2     Ontology metadata are accessible, even when the 
         → Ontology not found in a public registry
  [PASS] PURL1        F1     Ontology has a persistent URL
  [PASS] URI1         F1     Ontology URI is resolvable
  [FAIL] VER1         F1     A version IRI is declared in the ontology metada
         → Version IRI  not defined. Version info found (0.0.4).
  [FAIL] VER2         F1     Ontology version IRI resolves
         → Version IRI is not available, so it could not be resolved
  [FAIL] URI2         F1     Consistent ontology IDs are employed
         → Ontology URI is different fr

## Summary

In this notebook we made the Process Roles Ontology (PRO) FAIR-ready and published it:

| Step | What was added | FAIR dimension |
|---|---|---|
| 1 | `owl:imports <https://w3id.org/sulo>`, `owl:versionIRI`, `owl:versionInfo` | Interoperable — I2; Findable — F2 |
| 2 | Full ontology metadata (DC, VANN, PAV, DCAT, FOAF, MOD) | Reusable — R1 |
| 3 | Hand-written `rdfs:comment` definitions for every class | Reusable — R1.4 |
| 4 | Cleaned export to RDF/XML and Turtle | Accessible — A1 |
| 5 | Local FOOPS! pre-check | All |
| 6 | OWL 2 DL profile validation with ROBOT | Interoperable — I1 |
| 7 | Published to `https://w3id.org/ontostart/pro-{name}/` | All |
| 8 | Real FOOPS! API assessment on the live IRI | All |

Key takeaways:
- **FAIRness is transitive**: a domain ontology cannot be more FAIR than the extensions it imports. Publishing PRO at a persistent IRI removes a hidden blocker for any downstream ontology that uses PRO terms.
- **Small ontologies have no excuse for missing definitions** — every class in PRO can carry a hand-written `rdfs:comment` in minutes.
- **`owl:imports`** is the mechanism that makes external class references resolve under the OWL 2 DL profile. Without it, ROBOT flags every `sulo:Role` reference as undeclared.

---

## Exercises

### Exercise 1 — Validate the full imports closure

Re-run `robot validate-profile --profile DL` on your pizza ontology from notebook 07 (which imports PRO). Confirm that the eight previously reported `sulo:Role` / `sulo:Process` violations now disappear, because PRO itself now imports SULO.

In [10]:
# Exercise 1 — your shell command(s) here, or invoke ROBOT via subprocess

### Exercise 2 — Add a new role

PRO defines a `LocationRole` but no equivalent for *time*. Add a `TemporalRole` class as a subclass of `sulo:Role` with a clear `rdfs:label` and `rdfs:comment`, re-run the export, and verify the FAIRness pre-check still passes.

In [11]:
# Exercise 2 — your code here

### Exercise 3 — Reflect on stewardship

PRO is currently authored by a single person (you). For a community to adopt PRO, what additional metadata, governance signals, or external commitments would be required? Sketch a short plan.

*(Reflection — no code required.)*